# Leaf Disease Prediction — Retrain on Colab (GPU)

Final Year Project — Muhammad Asad Bukhari. Run the cells top to bottom:
**Runtime -> Run all**. Keep this tab open, training 10 epochs takes about 1-2 hours on a free T4 GPU.

Before running: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save**.

What this notebook does:
1. Checks the GPU
2. Clones the project from GitHub
3. Downloads the 1.7 GB dataset zip from Google Drive
4. Trains the custom CNN (10 epochs) and saves `trained_plant_disease_model.keras`
5. Evaluates on the validation set, plots curves, runs Grad-CAM
6. Saves the trained model + results to your Google Drive so nothing is lost

In [1]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
!git clone https://github.com/Asadshah-coder/leaf-disease-prediction.git
%cd leaf-disease-prediction
!pip install -q -r requirements.txt
!pip install -q gdown
!ls

Cloning into 'leaf-disease-prediction'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 52 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 2.68 MiB | 16.94 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/leaf-disease-prediction
disease_recognition.html  index.html  README.md		src
docs			  main.py     requirements.txt
home_page.png.png	  notebooks   results


In [3]:
# Dataset: Plant Disease FYP.zip (1.7 GB) from Google Drive, file ID 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx
!gdown --id 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx -O "Plant Disease FYP.zip"
!ls -lh "Plant Disease FYP.zip"
!unzip -q "Plant Disease FYP.zip"
!mkdir -p data
!mv "Plant Disease FYP/train" data/train
!mv "Plant Disease FYP/valid" data/valid
!echo "train images:"; find data/train -type f | wc -l
!echo "valid images:"; find data/valid -type f | wc -l
!echo "classes:"; ls data/train | wc -l

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=13_IbE48q2Nv3whIDxVf8OV60JzZGVySx
From (redirected): https://drive.google.com/uc?id=13_IbE48q2Nv3whIDxVf8OV60JzZGVySx&confirm=t&uuid=4f773a42-38c4-4f20-9109-bb98cb3c4f96
To: /content/leaf-disease-prediction/Plant Disease FYP.zip
100% 1.70G/1.70G [00:20<00:00, 83.1MB/s]
-rw-r--r-- 1 root root 1.6G Feb 13  2025 'Plant Disease FYP.zip'
train images:
70295
valid images:
17572
classes:
38


In [ ]:
# Train the custom CNN — about 1-2 hours on a T4 GPU. Do NOT close this tab.
!python src/train.py --data-dir data --epochs 10 --batch-size 32 --model-out trained_plant_disease_model.keras

Loading data from data ...
Found 70295 files belonging to 38 classes.
W0000 00:00:1793997833.131236     878 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Found 17572 files belonging to 38 classes.
Building the custom CNN (see src/model.py) ...
/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
Model: "sequential_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 1

In [ ]:
!python src/evaluate.py --model trained_plant_disease_model.keras --data-dir data/valid
!python src/plot_results.py
!ls -lh results/ trained_plant_disease_model.keras

In [ ]:
import json
print(open('results/evaluation.json').read())
print(open('results/classification_report.txt').read())

In [ ]:
# Grad-CAM on one sample image (first file in the Tomato Early blight folder)
!SAMPLE=$(ls "data/valid/Tomato___Early_blight" | head -n 1); echo "sample: $SAMPLE"; python src/gradcam.py --model trained_plant_disease_model.keras --image "data/valid/Tomato___Early_blight/$SAMPLE" --output results/gradcam.png
!ls -lh results/gradcam.png

In [ ]:
# Save the trained model + results to Google Drive (so they survive when this runtime ends)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/FYP-Leaf-Disease"
!cp trained_plant_disease_model.keras "/content/drive/MyDrive/FYP-Leaf-Disease/"
!cp -r results "/content/drive/MyDrive/FYP-Leaf-Disease/results"
!ls -lh "/content/drive/MyDrive/FYP-Leaf-Disease/"

## Done — next steps

1. The trained model is now in your Google Drive folder **FYP-Leaf-Disease** (`trained_plant_disease_model.keras`).
2. Copy the numbers from `results/evaluation.json` into the README results table (only quote numbers from this run).
3. Place the `.keras` file next to `main.py` and run `streamlit run main.py` to test the web app locally.
4. Push the new results to GitHub (`git add results/ README.md`, commit, push) — never commit the model file or the dataset, both are in `.gitignore`.